In [2]:
from google.colab import drive
import zipfile
import os

drive.mount('/content/drive')

# Your specific path
zip_path = '/content/drive/MyDrive/PlotQA-Stemsight/horizontal_bars_sample.zip'
extract_path = '/content/dataset_hbar'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ Extracted to {extract_path}")
!pip install -q transformers datasets sentencepiece accelerate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Extracted to /content/dataset_hbar


In [6]:
import json
from PIL import Image
from torch.utils.data import Dataset
from pathlib import Path

class HorizontalDonutDataset(Dataset):
    def __init__(self, dataset_path, processor, max_length=512, limit=10000):
        self.dataset_path = Path(dataset_path) / "train"
        self.processor = processor
        self.max_length = max_length

        metadata_path = self.dataset_path / "metadata.jsonl"

        if not metadata_path.exists():
            raise FileNotFoundError(f"❌ Missing metadata.jsonl at: {metadata_path}")

        with open(metadata_path, "r") as f:
            self.lines = f.readlines()[:limit]
        print(f"📋 Loaded {len(self.lines)} samples for Horizontal Bar training.")

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        line = json.loads(self.lines[idx])
        image_path = self.dataset_path / line["file_name"]
        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        target_sequence = f"<s_chartqa>{line['ground_truth']}</s_chartqa>"

        labels = self.processor.tokenizer(
            target_sequence,
            add_special_tokens=False,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values.squeeze(), "labels": labels.squeeze()}

In [11]:
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel, TrainingArguments, Trainer

# 1. Load Model & Processor (Starting from VBAR Master)
device = "cuda" if torch.cuda.is_available() else "cpu"
master_vbar_path = "/content/drive/MyDrive/STEM_Sight_VBAR_Final"

processor = DonutProcessor.from_pretrained(master_vbar_path)
model = VisionEncoderDecoderModel.from_pretrained(
    master_vbar_path,
    tie_word_embeddings=False
).to(device)

# 2. Prepare Dataset (10k Samples)
train_dataset = HorizontalDonutDataset("/content/dataset_hbar", processor, limit=10000)

# 🚀 CRITICAL FIX: Create a small validation subset (200 samples)
# This ensures "Validation Loss" appears INSTANTLY without freezing the training.
val_subset = torch.utils.data.Subset(train_dataset, range(0, 200))

# 3. Training Arguments (Optimized for Fast Feedback)
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/STEM_Sight_HBAR_Expert_V2",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_train_epochs=3,            # As requested: 3 Epochs
    weight_decay=0.01,

    # --- FEEDBACK SETTINGS ---
    logging_steps=10,              # Update Training Loss every 10 steps
    logging_first_step=True,       # Show results IMMEDIATELY at Step 1
    eval_strategy="steps",
    eval_steps=100,                # Show Validation Loss every 100 steps

    # --- PERFORMANCE ---
    fp16=True,                     # Use T4 GPU Speedup
    save_steps=500,                # Save every 500 steps
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=False   # Keep it fast
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_subset        # Fast validation on subset
)

print("🚀 Starting Training... Results will appear in the table in seconds.")
trainer.train()

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


📋 Loaded 10000 samples for Horizontal Bar training.
🚀 Starting Training... Results will appear in the table in seconds.


Step,Training Loss,Validation Loss
100,0.352604,0.057357
200,0.172445,0.040945
300,0.161842,0.030025
400,0.135655,0.021378
500,0.127927,0.020767
600,0.130604,0.013847
700,0.119647,0.010233
800,0.091132,0.009002
900,0.060498,0.007931
1000,0.093134,0.007778


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.18745066826343537, metrics={'train_runtime': 4838.0893, 'train_samples_per_second': 6.201, 'train_steps_per_second': 0.775, 'total_flos': 1.594731338661888e+19, 'train_loss': 0.18745066826343537, 'epoch': 3.0})